# Étape 2 : Parameter-Efficient Fine-Tuning (PEFT) avec LoRA

Ce notebook détaille la configuration de l'entraînement de `microsoft/phi-2`. 
**Contrainte technique :** Le passage d'un GPU de 6 Go à 24 Go de VRAM a permis d'abandonner la quantification au profit d'un chargement en `float16`, tout en ajustant la taille du batch pour accélérer l'itération.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model_id = "microsoft/phi-2"

print(f"Chargement du tokenizer et du modèle {model_id} en float16...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

## Configuration LoRA
Nous ciblons les projections d'attention et les couches denses pour maximiser l'adaptation tout en maintenant le nombre de paramètres entraînables sous la barre des 2%.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"]
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()